In [ ]:
import sys
import os
import matplotlib.pyplot as plt
sys.path.append(os.path.abspath('..'))

from src.config import ExperimentConfig, Hyperparameters
from src.trainer import Trainer
from src.inference import InferenceEngine
from src.models import JudgeModel

hyperparams = Hyperparameters(
    num_iterations=20,
    group_size=4,
    vllm_batch_size=2,
    learning_rate=5e-6,
    max_tokens=256,
    lora_rank=16,
    lora_target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    rl_coef=1.0,
    clip_ratio=0.2,
    temperature=0.8,
    top_p=0.9,
    gpu_memory_utilization=0.2
)

config = ExperimentConfig(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    judge_model="Qwen/Qwen2.5-Math-1.5B-Instruct",
    env="gsm8k",
    algo="rltf_fm",
    log_dir="./rltf_fm_alignment",
    hyperparameters=hyperparams
)

In [ ]:
trainer = Trainer(config)
losses, rewards = trainer.train()
del trainer

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(losses, marker='o', linestyle='-', color='b')
ax1.set_title('RLTF-FM Training Loss Curve')
ax1.set_xlabel('Iteration Step')
ax1.set_ylabel('Clipped Surrogate Loss')
ax1.grid(True)

ax2.plot(rewards, marker='s', linestyle='-', color='orange')
ax2.set_title('Training Average Reward (y0 Pass Rate)')
ax2.set_xlabel('Iteration Step')
ax2.set_ylabel('Reward (Accuracy)')
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
engine_base = InferenceEngine(config)
base_reward = engine_base.evaluate_env(num_samples=10)
del engine_base

In [ ]:
engine_tuned = InferenceEngine(config, adapter_path="./rltf_fm_alignment")
tuned_reward = engine_tuned.evaluate_env(num_samples=10)

In [ ]:
models = ['Base Qwen-0.5B', 'RLTF-FM Aligned Qwen']
eval_rewards = [base_reward, tuned_reward]

plt.figure(figsize=(6, 4))
bars = plt.bar(models, eval_rewards, color=['gray', 'orange'])
plt.title('Environment Pass Rate (Test Split Evaluation)')
plt.ylabel('Average Single-turn Reward')
plt.ylim(0, 1.05)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval, f"{yval*100:.1f}%", va='bottom', ha='center')

plt.show()

In [ ]:
prompt = "Let's think step by step. A store has 15 apples and sells 7. Then they receive a shipment of 20 apples. How many apples do they have now?"
print(f"Prompt: \n{prompt}\nBase Model response:")
engine_base.chat(prompt)
print("RLTF - Feedback Modelling Aligned Model response:")
engine_tuned.chat(prompt)

In [ ]:
judge = JudgeModel(config.judge_model, hyperparams=config.hyperparameters)
base_prompt = "Let's think step by step. A store has 15 apples and sells 7. Then they receive a shipment of 20 apples. How many apples do they have now?"
current_prompt = base_prompt

for round_num in range(1, 4): 
    print(f"\n--- Round {round_num} ---")
    response = engine_tuned._generate([current_prompt])[0]
    print(f"[RLTF-FM Aligned Model Answer]: {response}")
    
    if round_num < 3:
        critique = judge.get_critique(base_prompt, response)
        print(f"\n[Judge Critique]: {critique}")
        current_prompt = base_prompt + response + critique
